In [ ]:
import os
from odf.opendocument import load
from odf import table, text


def cell_text(cell):
    """Extract text from an ODS cell."""
    out = []
    for p in cell.getElementsByType(text.P):
        parts = []
        for node in p.childNodes:
            if hasattr(node, "data"):
                parts.append(node.data)
        out.append("".join(parts))
    return " ".join(out).strip()


def dm_to_dd(deg, minutes, west=False, south=False):
    """Convert degrees/minutes to decimal degrees."""
    dd = float(deg) + float(minutes) / 60.0
    if west or south:
        dd *= -1
    return dd

def check_val(val):
    if len(val) > 1:
        if  val[-1] == "S" or val[-1] == 'E':
            return val[0:-1]
    return val
    
def extract_positions_lons(ods_file):
    """Extract CTD surface, end-of-upcast positions, date and depth."""

    doc = load(ods_file)
    sheet = doc.spreadsheet.getElementsByType(table.Table)[0]
    rows = sheet.getElementsByType(table.TableRow)

    surface = None
    upcast = None
    date = None
    depth = None
    # print(rows)

    for i, row in enumerate(rows):
        # print(row)

        values = [cell_text(c) for c in row.getElementsByType(table.TableCell)]

        # ---------------- Date ----------------
        # Modify this string if your sheet uses a different label.
        # print(values)
        if values and "DATE" in values:
            coord_row = rows[i + 1]
            vals = [cell_text(c) for c in coord_row.getElementsByType(table.TableCell)]
            date = vals[2]

        # ---------------- Depth ----------------
        # Modify this string if your sheet uses a different label.
        if values and "Depth" in values:
            if len(values) > 1:
                depth = values[3]

        # ---------------- Surface ----------------
        if values and values[0] == "CTD Surface":

            coord_row = rows[i + 5]
            vals = [cell_text(c) for c in coord_row.getElementsByType(table.TableCell)]

            lat_deg = float(check_val(vals[0]))
            lat_min = float(check_val(vals[1]))
            lon_deg = float(check_val(vals[2]))
            lon_min = float(check_val(vals[3]))

            surface = {
                "lat_dd": dm_to_dd(lat_deg, lat_min, south=True),
                "lon_dd": dm_to_dd(lon_deg, lon_min),
                "lat_deg": lat_deg,
                "lat_min": lat_min,
                "lon_deg": lon_deg,
                "lon_min": lon_min,
            }

        # ---------------- End of upcast ----------------
        if values and values[0] == "CTD end of the upcast":

            coord_row = rows[i + 4]
            vals = [cell_text(c) for c in coord_row.getElementsByType(table.TableCell)]
            lat_deg = float(check_val(vals[0]))
            lat_min = float(check_val(vals[1]))
            lon_deg = float(check_val(vals[2]))
            lon_min = float(check_val(vals[3]))

            upcast = {
                "lat_dd": dm_to_dd(lat_deg, lat_min),
                "lon_dd": dm_to_dd(lon_deg, lon_min, west=True),
                "lat_deg": lat_deg,
                "lat_min": lat_min,
                "lon_deg": lon_deg,
                "lon_min": lon_min,
            }

    return surface, upcast

def get_next_value(values, key):
    for i, v in enumerate(values):
        if isinstance(v, str) and v.strip().startswith(key):
            if i + 1 < len(values):
                return values[i + 1]
    return None

import re
from datetime import datetime, time

import pandas as pd
from datetime import datetime

def parse_time(t):
    if t is None:
        return pd.NaT

    t = str(t).strip()

    if t == "":
        return pd.NaT

    t = t.replace("(local)", "").strip()

    if "h" in t:
        try:
            h, m = t.split("h")
            return datetime.strptime(f"{int(h):02d}:{int(m):02d}", "%H:%M")
        except:
            return pd.NaT

    for fmt in ("%H:%M:%S", "%H:%M"):
        try:
            return datetime.strptime(t, fmt)
        except ValueError:
            continue

    return pd.NaT
def extract_depths(ods_file):
    """Extract CTD surface, end-of-upcast positions, date and depth."""

    doc = load(ods_file)
    sheet = doc.spreadsheet.getElementsByType(table.Table)[0]
    rows = sheet.getElementsByType(table.TableRow)

    date = None
    depth = None
    time_start = None
    time_end = None

    for row in rows:

        values = [cell_text(c) for c in row.getElementsByType(table.TableCell)]

        # skip empty rows early
        if not values:
            continue

        if date is None and any(isinstance(v, str) and "Date" in v for v in values):
            date = get_next_value(values, "Date")

        if depth is None and any(isinstance(v, str) and "Depth" in v for v in values):
            depth = get_next_value(values, "Depth")

        if time_start is None and any(isinstance(v, str) and "Start sampling" in v for v in values):
            time_start = get_next_value(values, "Start sampling")

        if time_end is None and any(isinstance(v, str) and "End sampling" in v for v in values):
            time_end = get_next_value(values, "End sampling")

        # stop early if everything found
        if all(v is not None for v in [date, depth, time_start, time_end]):
            break
    return date, time_start, time_end, depth
def collect_ctd_casts(folder1, folder2):

    files_ctd = sorted([x for x in os.listdir(folder1) if x.startswith("W")])
    files_sample = sorted([x for x in os.listdir(folder2) if x.startswith("f")])

    casts = []

    cast_counter = 0

    for i in range(len(files_sample)):

        filepath = os.path.join(folder1, files_ctd[i])
        filepath_samples = os.path.join(folder2, files_sample[i])

        surface, upcast = extract_positions_lons(filepath)
        date, time_start, time_end, depth = extract_depths(filepath_samples)
        if time_start in ("", None) or time_end in ("", None):
            print(f"\nMissing start time in {files_sample[i]}")
            print(f"Date: {date}")
            print(f"Start time: {time_start}")
            print(f"End time: {time_end}")
            print(f"Depth: {depth}")

        cast_counter += 1

        casts.append({
            "cast_id": cast_counter,
            "lon_start": surface["lon_dd"],
            "lat_start": surface["lat_dd"],
            "lon_end": upcast["lon_dd"],
            "lat_end": upcast["lat_dd"],
            "depth": float(depth),
            "date": date,
            "time_start": parse_time(time_start),
            "time_end": parse_time(time_end),
        })

    return casts

In [17]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lon1, lat1, lon2, lat2):
    R = 6371  # km
    dlon = radians(lon2 - lon1)
    dlat = radians(lat2 - lat1)

    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return 2 * R * atan2(sqrt(a), sqrt(1 - a))


def assign_stations(casts, threshold_km=2):
    station_id = 1
    station_letters = {}

    prev_lon = None
    prev_lat = None
    letter_counter = {}

    for i, c in enumerate(casts):

        if i == 0:
            c["station_number"] = station_id
            letter_counter[station_id] = 0
            c["station_id"] = f"{station_id}a"
            prev_lon, prev_lat = c["lon_start"], c["lat_start"]
            continue

        dist = haversine(prev_lon, prev_lat, c["lon_start"], c["lat_start"])

        if dist < threshold_km:
            # same station
            c["station_number"] = station_id
        else:
            station_id += 1
            c["station_number"] = station_id
            letter_counter[station_id] = 0

        # station letter (a, b, c...)
        letter_counter[c["station_number"]] += 1
        letter = chr(96 + letter_counter[c["station_number"]])  # a,b,c...

        c["station_id"] = f"{c['station_number']}{letter}"

        prev_lon, prev_lat = c["lon_start"], c["lat_start"]

    return casts


In [18]:
import xarray as xr
import numpy as np

def save_to_nc(casts, outfile="ctd_metadata.nc"):

    n = len(casts)

    ds = xr.Dataset(
        data_vars=dict(
            station_number=("cast", [c["station_number"] for c in casts]),
            cast_id=("cast", [c["cast_id"] for c in casts]),
            depth=("cast", [c["depth"] for c in casts]),

            lon_start=("cast", [c["lon_start"] for c in casts]),
            lat_start=("cast", [c["lat_start"] for c in casts]),
            lon_end=("cast", [c["lon_end"] for c in casts]),
            lat_end=("cast", [c["lat_end"] for c in casts]),
            # time_start = ("cast", [c["time_start"] for c in casts]),
            # time_end = ("cast", [c["time_end"] for c in casts]),
        ),
        coords=dict(
            cast=np.arange(n),
            station_id=("cast", [c["station_id"] for c in casts]),
            time_start=("cast", [c["time_start"] for c in casts]),
            time_end=("cast", [c["time_end"] for c in casts]),
            date=("cast", [c["date"] for c in casts]),
        )
    )

    ds.to_netcdf(outfile)
    print(f"Saved: {outfile}")

In [19]:
sampling_path = "/Volumes/science/MD252_WHIRLS/CTD/sampling sheet/"


In [20]:
ctd_path = "/Volumes/science/MD252_WHIRLS/CTD/Station_sheet/"
casts = collect_ctd_casts(ctd_path, sampling_path)
casts = assign_stations(casts, threshold_km=2)
save_to_nc(casts)


Missing start time in fiche_ctd_015.ods
Date: 04/07/26
Start time: 
End time: 
Depth: 1200

Missing start time in fiche_ctd_017.ods
Date: 04/07/26
Start time: 
End time: 
Depth: 1200

Missing start time in fiche_ctd_019.ods
Date: 04/07/26
Start time: 18h30
End time: 
Depth: 1200
Saved: ctd_metadata.nc


In [13]:
import xarray as xr
path = "ctd_metadata.nc"
ds = xr.open_dataset(path)

In [15]:
ds.close()